# P1_00 · Colab setup: weights and data to Drive

Downloads and verifies the model weights (SAM ViT-B ≈ 375 MB, SAM ViT-H ≈ 2.4 GB, MedSAM ≈ 375 MB) and the datasets (Kvasir-SEG ≈ 46 MB, COCO val2017 annotations ≈ 250 MB, ISIC 2018 masks ≈ 27 MB, BUSI ≈ 195 MB) into `MyDrive/promptshift/`, and records the Colab software versions.

**Before you start** (once per Google account):
1. (Only if the repo is private again) Colab left bar → 🔑 *Secrets* → add `GH_TOKEN` (read access
   to `geekRoy0126/promptshift`), switch on *Notebook access*. Never paste a token into a cell.
2. *Runtime → Change runtime type → T4 GPU*.
3. *Runtime → Run all*. Allow Google Drive access when asked.

All data, weights and results go to `MyDrive/promptshift/`, so a disconnect loses nothing:
rerun the notebook and finished steps are skipped.

In [ ]:
# Parameters (edit before Run all if needed)
WEIGHTS = ["sam_vit_b", "sam_vit_h", "medsam"]
DATASETS = ["kvasir_seg", "coco_val2017", "isic2018", "busi"]

In [ ]:
# SETUP: identical in every notebook (generated by scripts/build_notebooks.py).
import base64
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/geekRoy0126/promptshift.git"
REF = "main"  # branch, tag or commit to run

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    REPO = Path("/content/promptshift")
    try:  # optional: the repo is public; GH_TOKEN is only needed if it is made private again
        _token = userdata.get("GH_TOKEN")
    except Exception:  # secret missing or notebook access not granted
        _token = ""
    _auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode() if _token else ""

    def _git(*args, auth=False):
        pre = ["-c", f"http.extraHeader=Authorization: Basic {_auth}"] if auth and _auth else []
        r = subprocess.run(["git", *pre, "-C", str(REPO), *args], capture_output=True, text=True)
        if r.returncode:
            msg = r.stderr or r.stdout
            for s in (_token, _auth):
                msg = msg.replace(s, "***") if s else msg
            raise RuntimeError(f"git {args[0]} failed: {msg}")

    REPO.mkdir(parents=True, exist_ok=True)
    if not (REPO / ".git").is_dir():
        _git("init", "-q")
    _git("fetch", "-q", "--depth", "1", REPO_URL, REF, auth=True)
    _git("reset", "-q", "--hard", "FETCH_HEAD")  # throwaway clone; all outputs live on Drive
    del _token, _auth
    _pip = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)]
    r = subprocess.run(_pip, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("pip install -e failed (log above)")
else:  # local CPU run: use the checkout this notebook lives in
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONUNBUFFERED"] = "1"
# scripts must import this checkout's code, whatever else is installed
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), os.environ.get("PYTHONPATH", "")])


def run(*args, check=True):
    """Run a repo script with live output; stop the notebook if it fails. Returns the output."""
    cmd = [sys.executable, *map(str, args)]
    print("$ python", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    while chunk := p.stdout.read1(4096):
        text = chunk.decode(errors="replace")
        out.append(text)
        sys.stdout.write(text)
        sys.stdout.flush()
    if p.wait() and check:
        raise RuntimeError(f"exit code {p.returncode}: python {' '.join(cmd[1:])}")
    return "".join(out)


def check_gpu():
    import torch

    if IN_COLAB and not torch.cuda.is_available():
        raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")


def git_out(*args):
    r = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)
    return r.stdout.strip()


SHA = git_out("rev-parse", "--short", "HEAD")
print(f"repo {REPO} @ {SHA} (Colab: {IN_COLAB})")

## 1. Environment
Saved to `MyDrive/promptshift/results/environment_colab.json`.

In [ ]:
import json
import platform

import numpy
import torch

from promptshift.paths import get_paths

paths = get_paths().ensure()
env = {
    "repo_sha": SHA,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": numpy.__version__,
}
print(json.dumps(env, indent=1))
(paths.results / "environment_colab.json").write_text(json.dumps(env, indent=1))
print("storage root:", paths.root)

## 2. Model weights
Checksums are verified after download (an existing file is only verified). If the MedSAM download from Google Drive fails with a quota error, download `medsam_vit_b.pth` by hand from the MedSAM README into `MyDrive/promptshift/weights/` and rerun this cell.

In [ ]:
run("scripts/download_weights.py", *[a for w in WEIGHTS for a in ("--model", w)])

## 3. Datasets
COCO and ISIC: annotations only (`--skip-images`); the few images the checks need are fetched one by one later (the ISIC image zip is 11 GB and only needed in P2). BUSI comes from Kaggle via `kagglehub` (public dataset, no key needed).

In [ ]:
for name in DATASETS:
    skip = ["--skip-images"] if name in ("coco_val2017", "isic2018") else []
    run("scripts/download_data.py", "--dataset", name, *skip)

## 4. What is on Drive now

In [ ]:
def size_mb(p):
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 2**20

for sub in (paths.weights, *sorted(paths.data.iterdir())):
    print(f"{size_mb(sub):9.1f} MB  {sub}")
print("\nP1_00 done. Next: P1_01_data_check.")